In [1]:
import json
import glob
import logging
from datetime import datetime

# Configure logging file to track output
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("fhir_light_validation.log"),
        logging.StreamHandler()
    ]
)

def load_fhir_resource(file_path):
    """
    Loads a JSON file and returns it as a Python dict.
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

def validate_patient(resource):
    errors = []
    warnings = []

    # Validate gender
    if resource.get("gender") not in ["male", "female", "other", "unknown"]:
        errors.append("Invalid gender")

    # Validate birthDate
    birth_date = resource.get("birthDate")
    if birth_date:
        try:
            datetime.strptime(birth_date, "%Y-%m-%d")
        except ValueError:
            errors.append("Invalid birthDate format")

    return errors, warnings

def validate_encounter(resource):
    errors = []
    warnings = []

    # Validate subject reference
    if not resource.get("subject") or not resource["subject"].get("reference"):
        errors.append("Missing subject reference")

    # Validate period
    period = resource.get("period")
    if period:
        start = period.get("start")
        end = period.get("end")
        if start and end:
            try:
                start_date = datetime.fromisoformat(start)
                end_date = datetime.fromisoformat(end)
                if start_date > end_date:
                    errors.append("period.start is after period.end")
            except ValueError:
                errors.append("Invalid period date format")

    return errors, warnings

def validate_bundle(resource):
    errors = []
    warnings = []

    entries = resource.get("entry", [])
    for entry in entries:
        subres = entry.get("resource")
        if subres:
            e, w = validate_resource(subres)
            errors.extend(e)
            warnings.extend(w)

    return errors, warnings

def validate_resource(resource):
    resource_type = resource.get("resourceType")
    if not resource_type:
        return ["Resource has no resourceType field"], []

    if resource_type == "Patient":
        return validate_patient(resource)
    elif resource_type == "Encounter":
        return validate_encounter(resource)
    elif resource_type == "Bundle":
        return validate_bundle(resource)
    else:
        return [], ["Resource type not specifically validated"]

# Adjust the path as needed
json_files = glob.glob("tests/synthea_sample_data_fhir_latest/*.json")

total_resources = 0
passed_resources = 0
failed_resources = 0

for file in json_files:
    resource = load_fhir_resource(file)
    total_resources += 1
    errors, warnings = validate_resource(resource)
    if errors:
        failed_resources += 1
        logging.error(f"Resource {total_resources} (type={resource.get('resourceType')}) => {len(errors)} errors, {len(warnings)} warnings")
        for error in errors:
            logging.error(f"  Error: {error}")
        for warning in warnings:
            logging.warning(f"  Warning: {warning}")
    else:
        passed_resources += 1
        logging.info(f"Resource {total_resources} (type={resource.get('resourceType')}) => {len(errors)} errors, {len(warnings)} warnings")
        for warning in warnings:
            logging.warning(f"  Warning: {warning}")

logging.info(f"Summary: {passed_resources} resources passed, {failed_resources} resources failed out of {total_resources} total resources")

2025-06-01 21:54:13,410 - INFO - Summary: 0 resources passed, 0 resources failed out of 0 total resources


In [2]:
# Generate sample FHIR data
import json
from datetime import datetime, timedelta
import random

def generate_sample_patient():
    return {
        "resourceType": "Patient",
        "id": f"patient-{random.randint(1000, 9999)}",
        "gender": random.choice(["male", "female", "other", "unknown"]),
        "birthDate": (datetime.now() - timedelta(days=random.randint(365*20, 365*80))).strftime("%Y-%m-%d"),
        "active": True
    }

def generate_sample_encounter(patient_id):
    start_date = datetime.now() - timedelta(days=random.randint(1, 30))
    end_date = start_date + timedelta(days=random.randint(1, 5))
    return {
        "resourceType": "Encounter",
        "id": f"encounter-{random.randint(1000, 9999)}",
        "subject": {
            "reference": f"Patient/{patient_id}"
        },
        "period": {
            "start": start_date.isoformat(),
            "end": end_date.isoformat()
        },
        "status": "finished"
    }

# Generate sample data
sample_resources = []

# Create 3 patients
for _ in range(3):
    patient = generate_sample_patient()
    sample_resources.append(patient)
    
    # Create 2 encounters for each patient
    for _ in range(2):
        encounter = generate_sample_encounter(patient["id"])
        sample_resources.append(encounter)

# Introduce some validation errors
sample_resources.append({
    "resourceType": "Patient",
    "gender": "invalid",  # Invalid gender
    "birthDate": "2023-13-45"  # Invalid date
})

sample_resources.append({
    "resourceType": "Encounter",
    "subject": {},  # Missing reference
    "period": {
        "start": "2023-01-15T00:00:00",
        "end": "2023-01-01T00:00:00"  # End before start
    }
})

print("Generated sample resources:")
print(json.dumps(sample_resources[:2], indent=2))  # Show first two resources only

Generated sample resources:
[
  {
    "resourceType": "Patient",
    "id": "patient-8071",
    "gender": "unknown",
    "birthDate": "1975-11-11",
    "active": true
  },
  {
    "resourceType": "Encounter",
    "id": "encounter-7500",
    "subject": {
      "reference": "Patient/patient-8071"
    },
    "period": {
      "start": "2025-05-16T21:54:20.800998",
      "end": "2025-05-19T21:54:20.800998"
    },
    "status": "finished"
  }
]


In [3]:
# Now run validation on sample data
for resource in sample_resources:
    errors, warnings = validate_resource(resource)
    if errors:
        logging.error(f"Resource (type={resource.get('resourceType')}) => {len(errors)} errors, {len(warnings)} warnings")
        for error in errors:
            logging.error(f"  Error: {error}")
        for warning in warnings:
            logging.warning(f"  Warning: {warning}")
    else:
        logging.info(f"Resource (type={resource.get('resourceType')}) => Passed validation")

2025-06-01 21:54:29,254 - INFO - Resource (type=Patient) => Passed validation
2025-06-01 21:54:29,258 - INFO - Resource (type=Encounter) => Passed validation
2025-06-01 21:54:29,259 - INFO - Resource (type=Encounter) => Passed validation
2025-06-01 21:54:29,260 - INFO - Resource (type=Patient) => Passed validation
2025-06-01 21:54:29,261 - INFO - Resource (type=Encounter) => Passed validation
2025-06-01 21:54:29,262 - INFO - Resource (type=Encounter) => Passed validation
2025-06-01 21:54:29,262 - INFO - Resource (type=Patient) => Passed validation
2025-06-01 21:54:29,263 - INFO - Resource (type=Encounter) => Passed validation
2025-06-01 21:54:29,264 - INFO - Resource (type=Encounter) => Passed validation
2025-06-01 21:54:29,264 - ERROR - Resource (type=Patient) => 2 errors, 0 warnings
2025-06-01 21:54:29,264 - ERROR -   Error: Invalid gender
2025-06-01 21:54:29,265 - ERROR -   Error: Invalid birthDate format
2025-06-01 21:54:29,265 - ERROR - Resource (type=Encounter) => 2 errors, 0 war

# Understanding FHIR Validation - The Simple Version! 🎮

Imagine you're checking a video game character creation form:

1. For a Patient, we check:
   - Gender must be one of: "male", "female", "other", "unknown" (like choosing your character type)
   - Birthday must be a real date (like 2000-12-25, not 2000-13-45)

2. For an Encounter (like when your character visits a doctor):
   - Must say which patient it belongs to (like having your player name)
   - Start time must be before end time (like a game level - can't finish before you start!)

When we run our checker:
- ✅ = Everything looks good!
- ❌ = Something's wrong (we'll tell you what)
- ⚠️ = Not perfect, but okay (like a warning message in a game)

In [4]:
# Let's check one patient record in a simple way
sample_patient = {
    "resourceType": "Patient",
    "gender": "dragon",  # Oops! This isn't valid
    "birthDate": "2000-01-01"  # This is good!
}

def simple_check_patient(patient):
    print(f"Checking patient record...\n")
    
    # Check gender
    valid_genders = ["male", "female", "other", "unknown"]
    if patient["gender"] in valid_genders:
        print("✅ Gender is valid!")
    else:
        print(f"❌ Oops! '{patient['gender']}' is not a valid gender.")
        print(f"   Must be one of these: {valid_genders}")
    
    # Check birthday
    try:
        datetime.strptime(patient["birthDate"], "%Y-%m-%d")
        print("✅ Birthday format is valid!")
    except:
        print("❌ Birthday must be like YYYY-MM-DD (example: 2000-01-01)")

print("Let's test our patient record:\n")
simple_check_patient(sample_patient)

Let's test our patient record:

Checking patient record...

❌ Oops! 'dragon' is not a valid gender.
   Must be one of these: ['male', 'female', 'other', 'unknown']
✅ Birthday format is valid!
